# 02 — Coleta de internações hospitalares (DATASUS SIH/SUS)

**Objetivo desta etapa:** obter, para os 645 municípios de SP, o número de
internações de idosos (60+) por quedas, fratura de fêmur, síncope e confusão
mental, entre 2019 e 2022 — a partir do SIH/SUS (Sistema de Informações
Hospitalares, grupo RD = AIH reduzida).

**Por que esse dado importa:** é a variável dependente da hipótese —
municípios com mais idosos sozinhos têm mais dessas internações?

**Saída desta etapa:** `data/processed/internacoes_sp.csv`, com uma linha por
(município, ano, causa) e a contagem de internações.

⚠️ **Este é o notebook mais demorado e o que mais costuma dar erro** — a
biblioteca `pysus` baixa arquivos grandes direto do FTP do DATASUS e a API
dela muda de versão para versão. **Rode célula por célula. No primeiro erro,
para e me manda a mensagem completa — não adianta tentar adivinhar sozinha,
eu ajusto o código na hora.**

Se o automático não der certo de jeito nenhum, a seção 2.4 explica o caminho
manual pelo TabNet.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd


## 2.1 Tentativa automática via `pysus`

Se o pacote não estiver instalado, rode no Anaconda Prompt:
```
pip install pysus
```

A API do `pysus` muda entre versões. Vamos tentar o caminho mais comum
(módulo `pysus.online_data.SIH`, baixa mês a mês); se a importação falhar,
a célula já avisa e pulamos para o caminho manual.


In [ ]:
try:
    from pysus.online_data.SIH import download as sih_download
    PYSUS_OK = True
except Exception as e:
    print("Não consegui importar pysus.online_data.SIH:", e)
    print("Tentando caminho alternativo (versões mais novas do pysus)...")
    PYSUS_OK = False

if not PYSUS_OK:
    try:
        from pysus.ftp.databases.sih import SIH as SIHClient
        PYSUS_OK = True
        PYSUS_NOVO = True
    except Exception as e:
        print("Também não consegui importar pysus.ftp.databases.sih:", e)
        print(">>> Pule para a seção 2.4 (download manual pelo TabNet) e me avise deste erro.")
        PYSUS_OK = False
        PYSUS_NOVO = False
else:
    PYSUS_NOVO = False


## 2.2 Download por ano/mês (grupo RD — AIH reduzida)

Isso pode demorar bastante (são 12 meses × 4 anos = 48 arquivos, para o
estado inteiro de SP). Se travar por muito tempo (mais de 15-20 minutos sem
progresso) ou der erro de conexão, interrompa e me avise — não precisa
insistir sozinha.


In [ ]:
aihs = []

if PYSUS_OK and not PYSUS_NOVO:
    for ano in config.ANOS_SIH:
        for mes in range(1, 13):
            try:
                df_mes = sih_download(config.UF_SIGLA, ano, mes)
                aihs.append(df_mes)
                print(f"OK {ano}-{mes:02d}: {len(df_mes)} registros")
            except Exception as e:
                print(f"FALHOU {ano}-{mes:02d}: {e}")

elif PYSUS_OK and PYSUS_NOVO:
    sih = SIHClient().load()
    arquivos = sih.get_files(group=["RD"], uf=config.UF_SIGLA, year=config.ANOS_SIH)
    baixados = sih.download(arquivos)
    for arq in baixados:
        try:
            aihs.append(arq.to_dataframe())
        except Exception as e:
            print(f"Falha ao converter {arq}: {e}")

else:
    print("pysus indisponível — vá para a seção 2.4.")

if aihs:
    df_sih = pd.concat(aihs, ignore_index=True)
    print(f"Total: {len(df_sih)} internações brutas (todas as idades, todas as causas)")
    df_sih.to_parquet(config.DATA_RAW / "sih_sp_bruto.parquet")
else:
    df_sih = None


## 2.3 Filtrar idosos e classificar por causa

As colunas do SIH que vamos usar (podem vir com nome ligeiramente diferente
dependendo da versão do pysus — **se der `KeyError`, me manda `df_sih.columns.tolist()`
que eu ajusto**):
- `IDADE` — idade do paciente (checar `COD_IDADE`: 4 = anos; outros códigos são dias/meses/horas — filtre por `COD_IDADE == 4` antes de comparar em anos)
- `DIAG_PRINC` — CID-10 do diagnóstico principal
- `MUNIC_RES` — código do município de residência (6 dígitos, igual ao `codigo_datasus` de `municipios_sp.csv`)
- `ANO_CMPT` / `MES_CMPT` — ano/mês de competência (internação)


In [ ]:
if df_sih is not None:
    df = df_sih.copy()

    # idade em anos (ajuste o nome da coluna de código de idade se vier diferente)
    if "COD_IDADE" in df.columns:
        df = df[df["COD_IDADE"].astype(str) == "4"]
    df["IDADE"] = pd.to_numeric(df["IDADE"], errors="coerce")
    df_idosos = df[df["IDADE"] >= config.IDADE_MINIMA_IDOSO].copy()

    df_idosos["causa"] = df_idosos["DIAG_PRINC"].apply(config.classificar_causa)
    df_idosos = df_idosos[df_idosos["causa"] != "outras"]

    df_idosos["codigo_datasus"] = df_idosos["MUNIC_RES"].astype(str).str[:6]
    df_idosos["ano"] = df_idosos["ANO_CMPT"].astype(int)

    internacoes = (
        df_idosos.groupby(["codigo_datasus", "ano", "causa"])
        .size()
        .reset_index(name="internacoes")
    )
    print(internacoes.shape)
    internacoes.head()
else:
    internacoes = None
    print("df_sih vazio — use a seção 2.4 antes de continuar.")


## 2.4 Caminho manual (TabNet) — use se o automático falhar

1. Acesse: http://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/mrsp.def
2. **Linha:** Município
3. **Coluna:** Ano processamento
4. **Conteúdo:** Internações
5. **Período:** 2019, 2020, 2021, 2022
6. Na seleção de **Faixa etária**, marque todas as faixas de 60 anos ou mais
7. Repita a consulta uma vez para cada causa (ou filtre por **Lista Morb. CID-10**
   pelos capítulos/categorias equivalentes a quedas W00-W19, fratura de fêmur
   S72, síncope R55, confusão mental R41/F05)
8. Exporte cada consulta em **CSV** e salve em `data/external/`, por exemplo:
   `sih_sp_manual_quedas.csv`, `sih_sp_manual_fratura_femur.csv`, etc.

Quando tiver os arquivos, me avisa que eu escrevo a célula de leitura — o
formato exportado pelo TabNet costuma vir "largo" (um ano por coluna,
separador `;`, encoding `latin1`) e precisa ser reorganizado para o mesmo
formato (`codigo_datasus`, `ano`, `causa`, `internacoes`) da seção 2.3, antes
de seguirmos para o notebook 03.


## 2.5 Salvar resultado consolidado


In [ ]:
if internacoes is not None:
    internacoes.to_csv(config.DATA_PROCESSED / "internacoes_sp.csv", index=False)
    print("Salvo em", config.DATA_PROCESSED / "internacoes_sp.csv")
else:
    print("Nada para salvar ainda — complete a seção 2.3 ou 2.4 primeiro.")
